# முன்னுரிமை உறுப்பினர் மிடில்வேர் உடன் ஹோட்டல் முன்பதிவு

இந்த நோட்புக் **நிகழ்பாட்டுக்குட்பட்ட மிடில்வேர்** Microsoft Agent Framework-ஐப் பயன்படுத்தி காட்டுகிறது. நாங்கள் பூர்வ-கூறிய முயற்சி எண்கலை மாதிரியைக் கொண்டு, முன்னுரிமை உறுப்பினர்கள் சிறப்பு அனுமதிகள் பெறும் ஒரு மிடில்வேர் அடுக்கு சேர்க்கிறோம்.

## நீங்கள் கற்பது:
1. **நிகழ்பாட்டுக்குட்பட்ட மிடில்வேர்**: செயல்பாடு முடிவுகளை தடுக்கவும் மாற்றவும்
2. **சூழல் அணுகல்**: செயல்பாடு முடிந்த பிறகும் `context.result`ஐப் படித்து மாற்றவும்
3. **வியாபாரக் கோட்பாடு செயல்படுத்தல்**: முன்னுரிமை உறுப்பினர் நன்மைகள்
4. **முடிவு மாற்றம்**: பயனர் நிலையைப் பொருத்து செயல்பாடு முடிவுகளை மாற்றுதல்
5. **ஏதேனும் workflow, வேறு முடிவுகள்**: மிடில்வேர் வழியாக நடத்தையை மாற்றுதல்

## மிடில்வேர் உடன் Workflow معماری:

```
User Input: "I want to book a hotel in Paris"
                    ↓
        [availability_agent]
        - Calls hotel_booking tool
        - 🌟 priority_check middleware intercepts
        - Checks user membership status
        - IF priority + no rooms → Override to available!
        - Returns BookingCheckResult
                    ↓
        Conditional Routing
           /                    \
    [has_availability]    [no_availability]
          ↓                      ↓
    [booking_agent]        [alternative_agent]
    (Priority override!)   (Regular users)
          ↓                      ↓
       [display_result executor]
```

## சூழலாலான workflow இலிருந்து முக்கிய வேறுபாடு:

**மிடில்வேர் இல்லாமல்** (14-conditional-workflow.ipynb):
- பாரிஸ்-ல் அறை இல்லை → alternative_agent-க்கு வழிமாறு

**மிடில்வேர் உடன்** (இந்த நோட்புக்):
- பொதுவான பயனர் + பாரிஸ் → அறை இல்லை → alternative_agent-க்கு வழிமாறு
- முன்னுரிமை பயனர் + பாரிஸ் → 🌟 மிடில்வேர் மேலோட்டம் செய்கிறது! → கையிருப்பு உள்ளது → booking_agent-க்கு வழிமாறு

## தேவைகள்:
- Microsoft Agent Framework நிறுவியிருத்தல்
- சூழலாலான workflow-ஐப் புரிந்துகொண்டிருத்தல் (14-conditional-workflow.ipynb -ஐ பாருங்கள)
- GitHub டோக்கன் அல்லது OpenAI API சாவி
- மிடில்வேர் மாதிரிகள் பற்றிய அடிப்படை புரிதல்


In [ ]:
import asyncio
import json
import os
from collections.abc import Awaitable, Callable
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    FunctionInvocationContext,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")

## படி 1: கட்டமைக்கப்பட்ட வெளியீடுகளுக்கான Pydantic மாதிரிகளை வரையறு

இந்த மாதிரிகள் முகவர்கள் திருப்பும் **வரையறை**யை வரையறுக்கின்றன. மிடில்வேர் கிடைப்பின் முடிவை மாற்றும் போது கண்காணிக்க `priority_override` என்பதை நாங்கள் சேர் செய்துள்ளோம்.


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str
    # Tracks if middleware overrode the result. The Azure structured-output
    # contract requires every property to be in the JSON schema's `required`
    # array, so we cannot give this a default value the way the original
    # notebook did.
    priority_override: bool


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check with priority_override)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## படி 2: முக்கிய உறுப்பினர்கள் தரவுத்தளத்தை வரையறுக்கவும்

இந்த டெமோவிற்கு, நாம் முக்கிய உறுப்பினர் தரவுத்தளத்தை சிமுலேட் செய்ய போகிறோம். உற்பத்தியில், இது உண்மையான தரவுத்தளம் அல்லது API-ஐ வினவுவதாக இருக்கும்.

**முக்கிய உறுப்பினர்கள்:**
- `alice@example.com` - VIP உறுப்பினர்
- `bob@example.com` - பிரீமியம் உறுப்பினர்  
- `priority_user` - சோதனை கணக்கு


In [ ]:
# Simulated priority members database
PRIORITY_MEMBERS = {
    "alice@example.com",
    "bob@example.com",
    "priority_user",
}

# Global variable to track current user (in real app, use proper session management)
current_user_id = "regular_user"  # Default: regular user


def set_user(user_id: str):
    """Set the current user for the session."""
    global current_user_id
    current_user_id = user_id
    is_priority = user_id in PRIORITY_MEMBERS
    status = "🌟 PRIORITY MEMBER" if is_priority else "👤 Regular User"

    display(
        HTML(f"""
        <div style='padding: 15px; background: {"linear-gradient(135deg, #FFD700 0%, #FFA500 100%)" if is_priority else "#e3f2fd"}; 
                    border-left: 4px solid {"#FF6B35" if is_priority else "#2196f3"}; border-radius: 4px; margin: 10px 0;'>
            <strong>👤 Current User Set:</strong> {user_id}<br>
            <strong>Status:</strong> {status}
        </div>
    """)
    )


print("✅ Priority members database created")
print(f"   Priority members: {len(PRIORITY_MEMBERS)} users")

## படி 3: ஹோட்டல் முன்பதிவு கருவியை உருவாக்கவும்

நிபந்தனை வேலைப_FLOW போலே, ஆனால் இப்போது அது மிடில்வேர் மூலம் இடையூறாக இருக்கும்!


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## படி 4: 🌟 முன்னுரிமை சரிபார்ப்பு மிடில்வேர் உருவாக்கவும் (முக்கிய அம்சம்!)

இது இந்த நோட்புக்கின் **முக்கிய செயல்பாடு** ஆகும். இந்த மிடில்வேர்:

1. **பிடிக்கும்** hotel_booking செயல்பாடு அழைப்பை
2. `next(context)` அழைப்பினால் செயல்பாட்டை வழக்கமான முறையில் **நிர்வகிக்கும்**
3. `context.result` உள்ள முடிவை **பரிசீலிக்கும்**
4. பயனர் முன்னுரிமை பெற்றவராவும் அறைகள் கிடைக்காமையும் இருந்தால் முடிவை **மீட்டமைக்கும்**
5. மாற்றியமைக்கப்பட்ட முடிவை முகவருக்கு **வரம்பாக்கும்**

**முக்கிய வடிவம்:**
```python
async def my_middleware(context, next):
    await next(context)  # செயல்பாட்டை இயக்கவும்
    # இப்போது context.result செயல்பாட்டின் வெளியீட்டை கொண்டுள்ளது
    if some_condition:
        context.result = new_value  # மேலதிகமாக எழுதவும்!
```


In [ ]:
async def priority_check_middleware(
    context: FunctionInvocationContext,
    next: Callable[[FunctionInvocationContext], Awaitable[None]],
) -> None:
    """
    Function middleware that overrides hotel_booking results for priority members.
    
    Workflow:
    1. Let the function execute normally
    2. Check if user is a priority member
    3. If priority + no availability → Override to make rooms available!
    4. Agent will then route to booking path instead of alternative path
    """
    function_name = context.function.name

    display(
        HTML(f"""
        <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Middleware:</strong> Intercepting {function_name}...
        </div>
    """)
    )

    # Execute the original function
    await next(context)

    # Now inspect and potentially modify the result
    if context.result and function_name == "hotel_booking":
        result_data = json.loads(context.result)
        destination = result_data.get("destination", "")
        has_availability = result_data.get("has_availability", False)

        # Check if user is priority member
        is_priority = current_user_id in PRIORITY_MEMBERS

        # Override logic: Priority member + no availability → Make available!
        if is_priority and not has_availability:
            display(
                HTML(f"""
                <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); 
                            border-radius: 8px; margin: 10px 0; box-shadow: 0 4px 12px rgba(255,165,0,0.4);'>
                    <h3 style='margin: 0 0 10px 0; color: #333;'>🌟 PRIORITY OVERRIDE ACTIVATED! 🌟</h3>
                    <p style='margin: 0; color: #555; line-height: 1.6;'>
                        <strong>User:</strong> {current_user_id}<br>
                        <strong>Status:</strong> VIP Priority Member<br>
                        <strong>Action:</strong> Overriding "No Availability" for {destination}<br>
                        <strong>Result:</strong> ✅ Rooms now available for priority booking!
                    </p>
                </div>
            """)
            )

            # Override the result!
            result_data["has_availability"] = True
            result_data["priority_override"] = True
            context.result = json.dumps(result_data)

        elif not has_availability:
            display(
                HTML(f"""
                <div style='padding: 12px; background: #ffebee; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>ℹ️ Middleware:</strong> No priority override (user: {current_user_id})
                </div>
            """)
            )


print("✅ priority_check_middleware created")
print("   - Intercepts hotel_booking function")
print("   - Overrides availability for priority members")

## படி 5: வழிநடத்துவதற்காக நிலைமைக் கர்சிகள் வரையறு

பொறுத்தொழில் வழிகாட்டியிடமுள்ள நிலைமைக் கர்சிகள் அதேவை - அவை கட்டமைக்கப்பட்ட வெளியீட்டை பரிசோதித்து வழிநடத்துதலை தீர்மானிக்கின்றன.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available (including priority overrides!)."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        # Check if this was a priority override
        override_indicator = " 🌟" if result.priority_override else ""

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}{override_indicator}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception:
        return False


print("✅ Condition functions defined")

## படி 6: தனிப்பயன் காட்சி செயற்பாட்டாளரை உருவாக்கவும்

முந்தைய செயற்பாட்டாளரே - இறுதி வேலைவாய்ப்பு வெளியீட்டை காட்சிப்படுத்துகிறது.


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """Display the final result as workflow output."""
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created")

## Step 7: சூழல் மாறிகளை ஏற்றவும்

LLM கிளையண்டை (GitHub மாதிரிகள் அல்லது OpenAI) கட்டமைக்கவும்.


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## படி 8: மிடில்வேர் உடன் AI முகவர்களை உருவாக்கவும்

**முக்கிய வேறுபாடு:** availability_agent உருவாக்கும்போது, நாம் `middleware` அளவுருவை கொடுக்கிறோம்!

இது எப்படி priority_check_middleware ஐ முகவரியின் செயல்பாடு அழைப்புக் குழாயில் சேர்க்கிறோம் என்பதைக் காட்டுகிறது.


In [ ]:
# Agent 1: Check availability with tool + middleware
availability_agent = AgentExecutor(
    await provider.create_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), message (string), "
            "and priority_override (bool, true if priority member got special access). "
            "The message should summarize the availability status and mention if priority override occurred."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
        middleware=[priority_check_middleware],  # 🌟 MIDDLEWARE INJECTION!
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    await provider.create_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    await provider.create_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "If priority_override is true in the input, mention that they received priority member access. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - WITH priority_check_middleware 🌟</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## படி 9: பணியினை கட்டமைக்கவும்

முந்தய பணிவழக் கடிகார வடிவமைப்பே - கிடைக்கும் இருப்பின்படி நிபந்தனையற்ற வழிசெலுத்தல்.


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH (can be triggered by middleware override!)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing (Middleware-Aware):</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> (or 🌟 <strong>priority override</strong>) → booking_agent → display_result
        </p>
    </div>
""")
)

## படி 10: சோதனை விடயம் 1 - பாரிஸில் உள்ள சாதாரண பயனர் (மேலோட்ட மாற்றமில்லை)

ஒரு சாதாரண பயனர் பாரிஸை பதிவு செய்ய முயற்சிக்கிறார் → அறைகள் இல்லை → வழிகள் alternative_agent க்கு செல்லும்


In [ ]:
# Set as regular user
set_user("regular_user")

display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Regular User + Paris</h3>
        <p style='margin: 0;'><strong>Expected:</strong> No rooms → No middleware override → Alternative suggestion</p>
    </div>
""")
)

# Create request
request_regular = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_regular = await workflow.run(request_regular)
outputs_regular = events_regular.get_outputs()

# Display results
if outputs_regular:
    result_regular = AlternativeResult.model_validate_json(outputs_regular[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: #fff; border: 2px solid #ff9800; border-radius: 12px; margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #e65100;'>📊 RESULT (Regular User)</h3>
            <div style='background: #fff3e0; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0;'><strong>Middleware:</strong> No priority override (regular user)</p>
                <p style='margin: 0 0 10px 0;'><strong>Alternative:</strong> 🏨 {result_regular.alternative_destination}</p>
                <p style='margin: 0;'><strong>Reason:</strong> {result_regular.reason}</p>
            </div>
        </div>
    """)
    )

## படி 11: சோதனை வழக்கு 2 - 🌟 முன்னுரிமை பயனர் பாரிஸில் (மேலோட்டம் உடன்!)

ஒரு முன்னுரிமை உறுப்பினர் பாற்ஸுக்கு முன்பதிவு செய்ய முயற்சிக்கிறார் → தொடக்கத்தில் அறைகள் கிடைக்கவில்லை → 🌟 மிடில்‌வேர் மாற்றி விடுகிறது! → booking_agent உடன் வழிமாற்றுகிறது

**இது மிடில்‌வேர் சக்தியின் முக்கிய காட்சி!**


In [ ]:
# Set as priority user
set_user("priority_user")

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #333;'>🧪 TEST CASE 2: 🌟 Priority User + Paris</h3>
        <p style='margin: 0; color: #555;'><strong>Expected:</strong> No rooms → 🌟 MIDDLEWARE OVERRIDE → Rooms available → Booking suggestion!</p>
    </div>
""")
)

# Create request
request_priority = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_priority = await workflow.run(request_priority)
outputs_priority = events_priority.get_outputs()

# Display results
if outputs_priority:
    result_priority = BookingConfirmation.model_validate_json(outputs_priority[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px;
                    box-shadow: 0 8px 16px rgba(255,165,0,0.4); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 RESULT (Priority Member) 🌟</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Priority Override!)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> 🌟 OVERRIDE ACTIVATED!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_priority.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_priority.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_priority.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #fff3cd; border-radius: 6px; border-left: 4px solid #FF6B35;'>
                    <strong>💡 What Just Happened:</strong><br>
                    1. hotel_booking tool returned "no availability"<br>
                    2. priority_check_middleware intercepted the result<br>
                    3. Middleware checked user status: priority_user ✅<br>
                    4. Middleware OVERRODE the result to "available"<br>
                    5. Workflow routed to booking_agent instead of alternative_agent!
                </div>
            </div>
        </div>
    """)
    )

## படி 12: சோதனை வழக்கு 3 - ஸ்டாக்ஹோமில் முன்னுரிமை பயனர் (ஏற்கனவே கிடைக்கும்)

முன்னுரிமை பயனர் ஸ்டாக்ஹோம் முயற்சி செய்கிறார் → அறைகள் கிடைக்கின்றன → மாற்றம் தேவையில்லை → booking_agent க்கு வழிமறைவு

தேவையான போது மட்டுமே பயன்முறை செயல்படுகிறது என்று இது காட்டுகிறது!


In [ ]:
# Priority user is still set from previous test

display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 3: Priority User + Stockholm</h3>
        <p style='margin: 0;'><strong>Expected:</strong> Rooms available → No override needed → Booking suggestion</p>
    </div>
""")
)

# Create request
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px;
                    box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 RESULT (Priority User - No Override Needed)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Natural)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> No override needed</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #e8f5e9; border-radius: 6px; border-left: 4px solid #4caf50;'>
                    <strong>💡 Middleware Behavior:</strong><br>
                    • hotel_booking returned "available" naturally<br>
                    • Middleware saw available = true → No override needed<br>
                    • Workflow proceeded normally to booking_agent
                </div>
            </div>
        </div>
    """)
    )

## முக்கியமான அம்சங்கள் மற்றும் மிடில்‌வேர் கருத்துகள்

### ✅ நீங்கள் கற்றுக்கொண்டது:

#### **1. செயல்பாடு அடிப்படையிலான மிடில்‌வேர் முறைமை**

மிடில்‌வேர் செயல்பாடு அழைப்புகளை எளிய ஆஸ்கிரவிலைப் பயன்படுத்தி இடையூறுப்படுத்துகிறது:

```python
async def my_middleware(
    context: FunctionInvocationContext,
    next: Callable,
) -> None:
    # செயல்பாடை தொடும்முன்
    print("Intercepting...")
    
    # செயல்பாட்டை செயற்படுத்தவும்
    await next(context)
    
    # செயல்பாடு முடிந்தபின் - முடிவைப் பரிசீலிக்கவும்
    if context.result:
        # தேவைப்பட்டால் முடிவில் மாற்றங்கள் செய்யவும்
        context.result = modified_value
```

#### **2. சூழல் அணுகல் மற்றும் முடிவை மாற்றல்**

- `context.function` - அழைக்கப்படும் செயல்பாட்டை அணுகுதல்
- `context.arguments` - செயல்பாடு முடிவுகளை வாசித்தல்
- `context.kwargs` - கூடுதல் அளவுருக்களை அணுகுதல்
- `await next(context)` - செயல்பாட்டை இயக்கு
- `context.result` - செயல்பாட்டின் வெளியீட்டை வாசித்தல்/மாற்றுதல்

#### **3. வணிக தர்க்க செயலாக்கம்**

எமது மிடில்‌வேர் முன்னுரிமை உறுப்பினர் நன்மைகளை நடைமுறைபடுத்துகிறது:
- **பொது பயனர்கள்**: மாற்றங்கள் இல்லை, சாதாரண பண்பொருள் ஓட்டம்
- **முன்னுரிமை பயனர்கள்**: "கிடைக்கவில்லை" → "கிடைக்கும்" என மாற்றல்
- **நிபந்தனை தர்க்கம்**: தேவையான நேரத்தில் மட்டுமே மாற்றல்

#### **4. அதே பண்பொருள் ஓட்டம், வேறுபட்ட முடிவுகள்**

மிடில்‌வேர் சக்தி:
- ✅ பண்பொருள் கட்டமைப்பில் மாற்றங்கள் இல்லை
- ✅ கருவி செயல்பாட்டில் மாற்றங்கள் இல்லை
- ✅ நிபந்தனை வழிமுறை தர்க்கத்தில் மாற்றங்கள் இல்லை
- ✅ வெறும் மிடில்‌வேர் → வேறுபட்ட நடத்தை!

### 🚀 உண்மை உலக பயன்பாடுகள்:

1. **VIP/பிரீமியம் அம்சங்கள்**
   - பிரீமியம் பயனர்களுக்கான வீத வரம்புகளை மாற்றல்
   - வளங்களுக்கு முன்னுரிமை அணுகலை வழங்கல்
   - பிரீமியம் அம்சங்களை இயங்குதளத்துடன் திறக்கல்

2. **A/B சோதனை**
   - பயனர்களை நேர்மாற்று செயல்பாடுகளுக்குக் வழிமாற்றுதல்
   - குறிப்பிட்ட பயனர்களுடன் புதிய அம்சங்களை சோதனை செய்தல்
   - படிப்படியாக அம்சங்களை வெளியீடு செய்தல்

3. **பாதுகாப்பு மற்றும் ஒத்துழைப்பு**
   - செயல்பாடு அழைப்புகளை ஆய்வு செய்தல்
   - நுண்ணறிவு செயல்பாடுகளை தடைசெய்தல்
   - வணிக விதிகளை அமல்படுத்தல்

4. **செயற்பாடு மேம்பாடு**
   - குறிப்பிட்ட பயனர்களுக்கான முடிவுகளை சேமித்தல்
   - சிக்கலான செயல்பாடுகளை தவிர்த்தல்
   - வள ஒதுக்கீடு தானாகச் செய்யல்

5. **பிழை கையாளுதல் மற்றும் மீண்டும் முயற்சி செய்தல்**
   - பிழைகளை பிடித்து நன்றாக கையாளுதல்
   - மீண்டும் முயற்சி தர்க்கத்தை செயல்படுத்தல்
   - மாற்று செயல்பாடுகளுக்குFallback செய்யல்

6. **பதிவு மற்றும் கண்காணிப்பு**
   - செயல்பாடு இயக்க நேரங்களை பதிவு செய்தல்
   - அளவுருக்கள் மற்றும் முடிவுகளை பதிவு செய்தல்
   - பயன்பாட்டுப் படிமாண்டல்களை கண்காணித்தல்

### 🔑 அலங்காரப்படுத்தலர்களிடமிருந்து முக்கிய வேறுபாடுகள்:

| அம்சம் | அலங்காரப்பொருள் | மிடில்‌வேர் |
|---------|----------------|------------|
| **பிரிவு** | தனி செயல்பாடு | முகவர் உள்ள அனைத்து செயல்பாடுகளும் |
| **வளைவுமுறை** | வரையறுக்கப்பட்ட பரிமாணம் | இயக்க நேரத்தில் மாற்றமுடியும் |
| **சூழல்** | வரம்பானது | முழு முகவர் சூழல் |
| **கூட்டு அமைப்பு** | பல அலங்காரங்கள் | மிடில்‌வேர் குழாய் |
| **முகவர் அறிவு** | இல்லை | உள்ளது (முகவர் நிலையை அணுகலாம்) |

### 📚 எப்போது மிடில்‌வேர் பயன்படுத்துவது:

✅ **மிடில்‌வேர் பயன்படுத்துகையில்:**
- பயனர்/அமர்வு நிலையின் அடிப்படையில் நடத்தையை மாற்ற வேண்டியிருந்தால்
- பல செயல்பாடுகளுக்கு தர்க்கம் விதிக்க வேண்டியிருந்தால்
- முகவர் நிலையில் சூழலை அணுக வேண்டியிருந்தால்
- குறுக்கு பரப்புகுறிப்பாக உள்ள கருத்துக்களை(பதிவு, அங்கீகாரம்) செயல்படுத்த வேண்டியிருந்தால்

❌ **மிடில்‌வேர் பயன்படுத்த வேண்டாம்:**
- எளிய உள்ளீடு சரிபார்ப்பு (Pydantic பயன்படுத்தவும்)
- செயல்பாடு-பிரத்தியேக தர்க்கம் (செயல்பாட்டில் வைத்திருங்கள்)
- ஒருமுறை மாற்றங்கள் (செயல்பாட்டை நேரடியாக மாற்றவும்)

### 🎓 மேம்பட்ட முறைமைகள்:

```python
# பல மிடில்வேர் (நிர்வகிப்பு வரிசை முக்கியம்!)
middleware=[
    logging_middleware,      # முதலில் பதிவு செய்க
    auth_middleware,         # பிறகு அங்கீகாரம் பரிசோதிக்கிறது
    cache_middleware,        # பிறகு கேஷை பரிசோதிக்கிறது
    rate_limit_middleware,   # பிறகு வீதத்தை வரையறுக்கிறது
    priority_check_middleware  # இறுதியாக முன்னுரிமை பரிசோதனை
]

# நிபந்தனை மிடில்வேர் செயல்பாடு
async def conditional_middleware(context, next):
    if should_execute(context):
        await next(context)
        # முடிவை மாற்றுக
    else:
        # செயல்பாட்டை முற்றிலும் தவிர்க்கவும்
        context.result = cached_value
```

### 🔗 தொடர்புடைய கருத்துகள்:

- **Agent Middleware**: agent.run() அழைப்புகளை இடையூறுபடுத்துகிறது
- **Function Middleware**: கருவி செயல்பாடு அழைப்புகளை இடையூறுபடுத்துகிறது (நாம் பயன்படுத்தியது!)
- **Middleware Pipeline**: வரிசையாக செயல்படும் மிடில்‌வேர் சங்கிலி
- **Context Propagation**: மிடில்‌வேர் சங்கிலி வழியாக நிலையைப் பரவச் செய்கிறது


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
